# Chapter 10: Design Patterns with First-Class Functions
*Fluent Python, 2nd Edition — Luciano Ramalho*

> "Conformity to patterns is not a measure of goodness." — Ralph Johnson

This notebook distills Chapter 10, showing how Python's first-class functions simplify classic GoF design patterns — specifically **Strategy** and **Command**.

## 1. Classic Strategy Pattern

The Strategy pattern defines a **family of algorithms**, encapsulates each one, and makes them interchangeable. The classic GoF version uses:

| Role | Class | Purpose |
|------|-------|--------|
| **Context** | `Order` | Uses a strategy to compute discounts |
| **Strategy** (ABC) | `Promotion` | Abstract interface with `discount()` |
| **Concrete Strategies** | `FidelityPromo`, `BulkItemPromo`, `LargeOrderPromo` | Each implements a discount rule |

### Discount rules
- **Fidelity**: 5% off for customers with >= 1000 fidelity points
- **Bulk item**: 10% off each line item with >= 20 units
- **Large order**: 7% off orders with >= 10 distinct items

In [ ]:
from abc import ABC, abstractmethod
from collections.abc import Sequence
from decimal import Decimal
from typing import NamedTuple, Optional


class Customer(NamedTuple):
    name: str
    fidelity: int


class LineItem(NamedTuple):
    product: str
    quantity: int
    price: Decimal

    def total(self) -> Decimal:
        return self.price * self.quantity


class Order(NamedTuple):  # the Context
    customer: Customer
    cart: Sequence[LineItem]
    promotion: Optional['Promotion'] = None

    def total(self) -> Decimal:
        totals = (item.total() for item in self.cart)
        return sum(totals, start=Decimal(0))

    def due(self) -> Decimal:
        if self.promotion is None:
            discount = Decimal(0)
        else:
            discount = self.promotion.discount(self)
        return self.total() - discount

    def __repr__(self):
        return f'<Order total: {self.total():.2f} due: {self.due():.2f}>'


class Promotion(ABC):  # the Strategy: an abstract base class
    @abstractmethod
    def discount(self, order: Order) -> Decimal:
        """Return discount as a positive dollar amount"""


class FidelityPromo(Promotion):
    """5% discount for customers with 1000 or more fidelity points"""
    def discount(self, order: Order) -> Decimal:
        rate = Decimal('0.05')
        if order.customer.fidelity >= 1000:
            return order.total() * rate
        return Decimal(0)


class BulkItemPromo(Promotion):
    """10% discount for each LineItem with 20 or more units"""
    def discount(self, order: Order) -> Decimal:
        discount = Decimal(0)
        for item in order.cart:
            if item.quantity >= 20:
                discount += item.total() * Decimal('0.1')
        return discount


class LargeOrderPromo(Promotion):
    """7% discount for orders with 10 or more distinct items"""
    def discount(self, order: Order) -> Decimal:
        distinct_items = {item.product for item in order.cart}
        if len(distinct_items) >= 10:
            return order.total() * Decimal('0.07')
        return Decimal(0)


# ---------- demo ----------
joe = Customer('John Doe', 0)
ann = Customer('Ann Smith', 1100)
cart = [
    LineItem('banana', 4, Decimal('.5')),
    LineItem('apple', 10, Decimal('1.5')),
    LineItem('watermelon', 5, Decimal(5)),
]

print('Joe (0 pts), FidelityPromo:', Order(joe, cart, FidelityPromo()))
print('Ann (1100 pts), FidelityPromo:', Order(ann, cart, FidelityPromo()))

banana_cart = [LineItem('banana', 30, Decimal('.5')),
               LineItem('apple', 10, Decimal('1.5'))]
print('Joe, BulkItemPromo:', Order(joe, banana_cart, BulkItemPromo()))

long_cart = [LineItem(str(sku), 1, Decimal(1)) for sku in range(10)]
print('Joe, LargeOrderPromo:', Order(joe, long_cart, LargeOrderPromo()))

### Key observation

Each concrete strategy is a **class with a single method** and **no instance state**. They look a lot like plain functions — and they *should be* plain functions.

## 2. Function-Oriented Strategy

Replace the ABC + subclasses with **plain functions**. The `Order` context accepts a `Callable[[Order], Decimal]` instead of a `Promotion` object.

**What changes:**
- No more `Promotion` ABC
- No more classes for each strategy — just functions
- `promotion` attribute is typed as `Optional[Callable[["Order"], Decimal]]`
- Call with `self.promotion(self)` instead of `self.promotion.discount(self)`

In [ ]:
from collections.abc import Sequence
from dataclasses import dataclass
from decimal import Decimal
from typing import Optional, Callable, NamedTuple


class Customer(NamedTuple):
    name: str
    fidelity: int


class LineItem(NamedTuple):
    product: str
    quantity: int
    price: Decimal

    def total(self) -> Decimal:
        return self.price * self.quantity


@dataclass(frozen=True)
class Order:  # the Context
    customer: Customer
    cart: Sequence[LineItem]
    promotion: Optional[Callable[['Order'], Decimal]] = None

    def total(self) -> Decimal:
        totals = (item.total() for item in self.cart)
        return sum(totals, start=Decimal(0))

    def due(self) -> Decimal:
        if self.promotion is None:
            discount = Decimal(0)
        else:
            discount = self.promotion(self)  # call the function directly
        return self.total() - discount

    def __repr__(self):
        return f'<Order total: {self.total():.2f} due: {self.due():.2f}>'


# ---------- Strategies as plain functions ----------

def fidelity_promo(order: Order) -> Decimal:
    """5% discount for customers with 1000 or more fidelity points"""
    if order.customer.fidelity >= 1000:
        return order.total() * Decimal('0.05')
    return Decimal(0)


def bulk_item_promo(order: Order) -> Decimal:
    """10% discount for each LineItem with 20 or more units"""
    discount = Decimal(0)
    for item in order.cart:
        if item.quantity >= 20:
            discount += item.total() * Decimal('0.1')
    return discount


def large_order_promo(order: Order) -> Decimal:
    """7% discount for orders with 10 or more distinct items"""
    distinct_items = {item.product for item in order.cart}
    if len(distinct_items) >= 10:
        return order.total() * Decimal('0.07')
    return Decimal(0)


# ---------- demo ----------
joe = Customer('John Doe', 0)
ann = Customer('Ann Smith', 1100)
cart = [
    LineItem('banana', 4, Decimal('.5')),
    LineItem('apple', 10, Decimal('1.5')),
    LineItem('watermelon', 5, Decimal(5)),
]

print('--- Function-Oriented Strategy ---')
print('Joe, fidelity_promo:', Order(joe, cart, fidelity_promo))
print('Ann, fidelity_promo:', Order(ann, cart, fidelity_promo))

banana_cart = [LineItem('banana', 30, Decimal('.5')),
               LineItem('apple', 10, Decimal('1.5'))]
print('Joe, bulk_item_promo:', Order(joe, banana_cart, bulk_item_promo))

long_cart = [LineItem(str(sku), 1, Decimal(1)) for sku in range(10)]
print('Joe, large_order_promo:', Order(joe, long_cart, large_order_promo))

### Why `self.promotion(self)`?

`promotion` is an **instance attribute** (a callable), not a method bound to the class. Python's descriptor protocol does not auto-bind it, so you must explicitly pass the `Order` instance.

## 3. Choosing the Best Strategy

Since strategies are now just functions, we can store them in a list and write a **meta-strategy** that picks the one yielding the largest discount.

In [ ]:
# Using the same Order, Customer, LineItem, and promo functions from above

promos = [fidelity_promo, bulk_item_promo, large_order_promo]


def best_promo(order: Order) -> Decimal:
    """Compute the best discount available"""
    return max(promo(order) for promo in promos)


# ---------- demo ----------
print('--- best_promo picks the winner ---')
print('Joe + long_cart:', Order(joe, long_cart, best_promo))     # large_order wins
print('Joe + banana_cart:', Order(joe, banana_cart, best_promo))  # bulk_item wins
print('Ann + cart:', Order(ann, cart, best_promo))                # fidelity wins

**Problem:** If you add a new promo function and forget to add it to `promos`, `best_promo` silently ignores it. The next sections solve this.

## 4. Finding Strategies via Module Introspection

Two automatic approaches to building the `promos` list:

### Approach A: `globals()` filtering
```python
promos = [promo for name, promo in globals().items()
          if name.endswith('_promo') and name != 'best_promo']
```

### Approach B: `inspect.getmembers()` on a dedicated module
```python
import inspect
import promotions  # a module containing only promo functions

promos = [func for _, func in inspect.getmembers(promotions, inspect.isfunction)]
```

Both eliminate manual list maintenance, but they rely on **naming conventions** or **module-level assumptions**.

In [ ]:
# Demonstrate approach A: globals() introspection
# We simulate it by filtering our current namespace

# Collect all functions ending in '_promo' (excluding best_promo)
promos_auto = [obj for name, obj in list(globals().items())
               if callable(obj)
               and name.endswith('_promo')
               and name != 'best_promo']

print('Auto-discovered promos:', [f.__name__ for f in promos_auto])

def best_promo_auto(order: Order) -> Decimal:
    """Compute the best discount using auto-discovered promos"""
    return max(promo(order) for promo in promos_auto)

print('Ann + cart (auto):', Order(ann, cart, best_promo_auto))

## 5. Decorator-Enhanced Strategy Pattern

The cleanest solution: a **registration decorator** that appends each strategy function to the list as it is defined. No naming conventions, no introspection — fully explicit.

In [ ]:
from collections.abc import Sequence
from dataclasses import dataclass
from decimal import Decimal
from typing import Optional, Callable, NamedTuple

# ---------- Core types (redefined for self-contained demo) ----------

class Customer(NamedTuple):
    name: str
    fidelity: int

class LineItem(NamedTuple):
    product: str
    quantity: int
    price: Decimal
    def total(self) -> Decimal:
        return self.price * self.quantity

Promotion = Callable[['Order'], Decimal]

promos: list[Promotion] = []

def promotion(promo: Promotion) -> Promotion:
    """Registration decorator: appends promo to the global list."""
    promos.append(promo)
    return promo  # returns the function unchanged

@dataclass(frozen=True)
class Order:
    customer: Customer
    cart: Sequence[LineItem]
    promo: Optional[Promotion] = None

    def total(self) -> Decimal:
        return sum((item.total() for item in self.cart), start=Decimal(0))

    def due(self) -> Decimal:
        if self.promo is None:
            return self.total()
        return self.total() - self.promo(self)

    def __repr__(self):
        return f'<Order total: {self.total():.2f} due: {self.due():.2f}>'


def best_promo(order: Order) -> Decimal:
    """Compute the best discount available"""
    return max(promo(order) for promo in promos)


# ---------- Decorated strategies ----------

@promotion
def fidelity(order: Order) -> Decimal:
    """5% discount for customers with 1000+ fidelity points"""
    if order.customer.fidelity >= 1000:
        return order.total() * Decimal('0.05')
    return Decimal(0)

@promotion
def bulk_item(order: Order) -> Decimal:
    """10% discount for each LineItem with 20+ units"""
    discount = Decimal(0)
    for item in order.cart:
        if item.quantity >= 20:
            discount += item.total() * Decimal('0.1')
    return discount

@promotion
def large_order(order: Order) -> Decimal:
    """7% discount for orders with 10+ distinct items"""
    distinct_items = {item.product for item in order.cart}
    if len(distinct_items) >= 10:
        return order.total() * Decimal('0.07')
    return Decimal(0)


# ---------- demo ----------
joe = Customer('John Doe', 0)
ann = Customer('Ann Smith', 1100)
cart = [LineItem('banana', 4, Decimal('.5')),
        LineItem('apple', 10, Decimal('1.5')),
        LineItem('watermelon', 5, Decimal(5))]

print('--- Decorator-Enhanced Strategy ---')
print('Registered promos:', [p.__name__ for p in promos])
print('Ann + cart, best_promo:', Order(ann, cart, best_promo))

banana_cart = [LineItem('banana', 30, Decimal('.5')),
               LineItem('apple', 10, Decimal('1.5'))]
print('Joe + banana_cart, best_promo:', Order(joe, banana_cart, best_promo))

### Advantages of the decorator approach
- **No naming convention** required (`_promo` suffix unnecessary)
- **Self-documenting**: `@promotion` makes intent clear at the definition site
- **Easy to toggle**: comment out the decorator to disable a strategy
- **Cross-module**: strategies can live anywhere, as long as they use `@promotion`

## 6. The Command Pattern with Callables

The **Command** pattern decouples the *invoker* (e.g. a menu item) from the *receiver* (the object doing the work) via a `Command` object with an `execute()` method.

In Python, any **callable** already implements a single-method interface: `__call__`. So commands can simply be functions, and a `MacroCommand` can be a class that holds a list of callables.

In [ ]:
class MacroCommand:
    """A command that executes a list of commands"""

    def __init__(self, commands):
        self.commands = list(commands)  # ensure iterable, keep local copy

    def __call__(self):
        for command in self.commands:
            command()


# ---------- demo: simulated editor commands ----------

class Document:
    def __init__(self, text=''):
        self.text = text
        self.clipboard = ''

    def __repr__(self):
        return f'Document({self.text!r})'


doc = Document('Hello, world!')

# Commands as plain functions (closures over doc)
def select_all():
    doc.clipboard = doc.text
    print(f'  Selected all: {doc.clipboard!r}')

def uppercase():
    doc.text = doc.text.upper()
    print(f'  Uppercased: {doc.text!r}')

def add_exclaim():
    doc.text = doc.text + '!!!'
    print(f'  Added !!!: {doc.text!r}')


# Individual command
print('--- Individual commands ---')
select_all()

# MacroCommand: batch multiple commands
print()
print('--- MacroCommand ---')
doc = Document('Hello, world!')   # reset
macro = MacroCommand([uppercase, add_exclaim, select_all])
macro()  # calls all three in sequence
print(f'Final document: {doc}')

### When do you still need a Command class?

- **Undo support**: the command needs to store state to reverse its action
- **Logging / replay**: the command object is serialized

Even then, consider **closures** or **callable instances** (classes with `__call__`) before building a full class hierarchy.

## Summary: The Big Idea

| Pattern | Classic GoF | Pythonic replacement |
|---------|------------|---------------------|
| **Strategy** | ABC + concrete subclasses | Plain functions (or callables) |
| **Command** | Command interface + subclasses | Functions / `__call__` objects |
| **Flyweight** (for strategies) | Shared strategy instances | Functions (created once at module load) |

**Core insight:** Whenever a design pattern requires single-method classes with no state, replace them with **first-class functions**. Python's callable protocol (`__call__`) means every function already implements a single-method interface.

> "16 of 23 patterns have qualitatively simpler implementation in Lisp or Dylan than in C++" — Peter Norvig (1996)

## Exercises

### Exercise 1: Add a new strategy
Add a `combo_promo` that gives 12% off when both conditions are met: the customer has >= 500 fidelity points AND the order has >= 5 distinct items.

In [ ]:
# Exercise 1 — Try it yourself, then check below


# YOUR CODE HERE


# ---------- Solution ----------
@promotion
def combo(order: Order) -> Decimal:
    """12% discount for 500+ fidelity AND 5+ distinct items"""
    distinct = len({item.product for item in order.cart})
    if order.customer.fidelity >= 500 and distinct >= 5:
        return order.total() * Decimal('0.12')
    return Decimal(0)

# Verify it was registered
print('Promos after adding combo:', [p.__name__ for p in promos])

# Test
loyal = Customer('Loyal Lucy', 800)
varied_cart = [LineItem(str(i), 2, Decimal('3.00')) for i in range(6)]
print('Loyal + varied_cart, combo:', Order(loyal, varied_cart, combo))
print('Loyal + varied_cart, best_promo:', Order(loyal, varied_cart, best_promo))

### Exercise 2: MacroCommand with undo
Extend `MacroCommand` to support an `undo()` method that reverses the commands.

In [ ]:
# Exercise 2 — MacroCommand with undo

class UndoableMacro:
    """MacroCommand that supports undo via paired (do, undo) callables."""

    def __init__(self, command_pairs):
        """command_pairs: list of (do_func, undo_func) tuples"""
        self.command_pairs = list(command_pairs)
        self.executed = []

    def __call__(self):
        for do_fn, undo_fn in self.command_pairs:
            do_fn()
            self.executed.append(undo_fn)

    def undo(self):
        """Undo in reverse order"""
        while self.executed:
            undo_fn = self.executed.pop()
            undo_fn()


# Demo
state = {'value': 0}

def add_ten():
    state['value'] += 10
    print(f"  +10 -> {state['value']}")

def sub_ten():
    state['value'] -= 10
    print(f"  -10 -> {state['value']}")

def double():
    state['value'] *= 2
    print(f"  *2  -> {state['value']}")

def halve():
    state['value'] //= 2
    print(f"  //2 -> {state['value']}")


macro = UndoableMacro([(add_ten, sub_ten), (double, halve)])

print('--- Execute ---')
macro()
print(f"State: {state['value']}")

print()
print('--- Undo ---')
macro.undo()
print(f"State: {state['value']}")